In [1]:
# -*- coding: utf-8 -*-
"""
Created on Wed Oct 21 14:12:42 2020

@author: clara, viggy
"""

import numpy as np
import matplotlib.pyplot as plt
import copy
import statistics
import pandas as pd
import os
import re
import glob
from scipy.signal import find_peaks
from func_stepFrequency import get_gait, get_peaks, calc_step_width, calculate_step_stride_length, calc_interstep_all, get_step_lengths, get_frames, variability, asymmetry, intersect, calculate_stance_swing, calculate_percent_cycle, double_support_time, apply_lowpass_filter
import traceback

In [2]:
new_frames = pd.DataFrame()
error_subj = []
error_trace = []
error_idx = []

In [3]:
INDICES = ["trial_num", "test_type",
                      "Avg_LF_stance_time", 
                      "Avg_LF_swing_time",
                      "Avg_RF_stance_time",
                      "Avg_RF_swing_time",
                      "Variability_LF_stance_time",
                      "Variability_LF_swing_time",
                      "Variability_RF_stance_time",
                      "Variability_RF_swing_time",
                      "Asymmetry_Stance_time",
                      "Asymmetry_Swings_time",
                      "total_%_swing_time",
                      "total_%_stance_time",
                      "double_support_time",
                      "Avg_InterStepTime",
                      "Variability_InterStepTime",
                      "Frequency",
                      "Variance_COM_y",
                      "speed",
                      "Avg_stride_length",
                      "Variance_stride_length",
                      "Variability_stride_length",
                      "Asymmetry_stride_length",
                      "Avg_step_length",
                      "Variance_step_length",
                      "Variability_step_length",
                      "Avg_step_width",
                      "Variance_step_width",
                      "Variability_step_width",
                      "Avg_LF_max_height",
                      "Avg_RF_max_height",
                      "Variability_LF_max_height",
                      "Variability_RF_max_height",
                      "Asymmetry_F_max_height"]

## Load COM Data

In [ ]:
#filelocation = glob.glob(f'../data/gang/*.txt')
#filelocation = glob.glob(f'Patient_txt_PD_reup/*.txt')
#filelocation = glob.glob(f'Patient_txt_ataxia_reup/*.txt')
#filelocation = glob.glob(f'missing_txt_files_20250818/*.txt')
filelocation = glob.glob(f'txt_healthy/*.txt')


indexes=['test_type','Step_Frequency', 'Av_Step_Length', 'Var_Step_Length',
         'Inter_Step_time', 'Var_InterStepTime', 'Speed', 'Var_COMy',
         'Step_width', 'LF_height', 'RF_height', 'Var_Step_width']
results = pd.DataFrame(index=indexes)

all_results = pd.DataFrame()
counter = 0
PATTERN = r'-[2-9]-'

for files in filelocation:
    try:
        data = np.loadtxt(files, encoding = 'latin1') 
        #sub_name = [s[0:6] for s in os.path.basename(files).split()]
        file_strip = os.path.basename(files)
        sub_name = file_strip[0:6]
        
        #search for multiple trials
        search = re.search(PATTERN, file_strip)
        if search != None: #match
            trial_num = search.group(0)[1]
        else: #no match
            trial_num = "1"
    
        subject = sub_name
        
        '''
        print("file_strip", file_strip)
        print("sub_name", sub_name)
        print("trial_num", trial_num)
        
        print(f'Starting analysis for subject: {subject}')
        print(f"---------index {counter}---------")
        '''
        
        if 'normalergang' in files.lower():
            test_type = 'NG'
            #sub_name += num
            #sub_name += "_NG"
            lanes = ['lane_1', 'lane_2', 'lane_3', 'lane_4', 'lane_5', 'lane_6']
        elif 'rückwärtsgang' in files.lower():
            test_type = 'RG'
            #sub_name += num
            #sub_name += '_RG'
            #lanes = ['lane_1', 'lane_2', 'lane_3', 'lane_4', 'lane_5', 'lane_6']
            lanes = ['lane_1', 'lane_2', 'lane_3', 'lane_4']
        elif 'tandemgang' in files.lower():
            test_type = 'TG'
            #sub_name += num
            #sub_name += '_TG'
            lanes = ['lane_1', 'lane_2', 'lane_3', 'lane_4']

        timeframes = data[:,1] #fractions of a second?
        time = timeframes / 60 #seconds or minutes?
        
        '''OLD VALUES
        COM_x = data [:,14] #sagittal / anterior posterior
        COM_y = data[:,15] # right left
        COM_z = data[:,16] # vertical / up down
        LeftFoot_x = data [:,17]
        LeftFoot_y = data [:,18]
        LeftFoot_z = data [:,19]
        RightFoot_x = data [:,20] #front-back in m
        RightFoot_y = data [:,21] #left-right
        RightFoot_z = data [:,22] #height
        '''

        COM_x = data [:,2] #sagittal / anterior posterior
        COM_y = data[:,3] # right left
        COM_z = data[:,4] # vertical / up down
        LeftFoot_x = data [:,11]
        LeftFoot_y = data [:,12]
        LeftFoot_z = data [:,13]
        RightFoot_x = data [:,14] #front-back in m
        RightFoot_y = data [:,15] #left-right
        RightFoot_z = data [:,16] #height

        # ---- Butterworth low-pass at construction (4th-order, 6 Hz, zero-phase) ----
        # Filter all position channels once here so every downstream consumer (get_peaks, the
        # stance/swing + DST path, and the x/y distance features: stride/step length, step
        # width, speed, COM_y variance) sees the same filtered signal. get_peaks runs with
        # filt=False (its default) to avoid double-filtering. Note: COM_x feeds gait-plate/lane
        # detection below, so that now also runs on filtered data (negligible on a slow path).
        COM_x = apply_lowpass_filter(COM_x)
        COM_y = apply_lowpass_filter(COM_y)
        COM_z = apply_lowpass_filter(COM_z)
        LeftFoot_x = apply_lowpass_filter(LeftFoot_x)
        LeftFoot_y = apply_lowpass_filter(LeftFoot_y)
        LeftFoot_z = apply_lowpass_filter(LeftFoot_z)
        RightFoot_x = apply_lowpass_filter(RightFoot_x)
        RightFoot_y = apply_lowpass_filter(RightFoot_y)
        RightFoot_z = apply_lowpass_filter(RightFoot_z)

        #print(f'Test Type: {test_type}')
        
        '''
        print("---- Walk plot ----")
        plt.axis('equal')
        plt.plot(LeftFoot_x, LeftFoot_y)
        plt.axis('equal')
        plt.plot(RightFoot_x, RightFoot_y)
        plt.axis('equal')
        plt.plot(COM_x, COM_y)
        plt.show()
        #plt.plot(time, LeftFoot_y, time, COM_y, time, RightFoot_y)
        '''

        COM_z_time = np.array([time]+[COM_z]) #COM_z values with corresponding time
        COM_x_time = np.array([time]+[COM_x])
        COM_y_time = np.array([time]+[COM_y])

        
        ## Calculate Time at Gait Plate
        maxCOMx = max(COM_x)-1
        minCOMx = min(COM_x)+1


        COM_x_new = copy.copy(COM_x)
        COM_x_new[COM_x_new > maxCOMx] = 9.999
        COM_x_new[COM_x_new < minCOMx] = 9.999
        COM_x_new_time = np.array([time] + [COM_x_new])
        
        #PRINT CENTER OF MASS GRAPHS
        #print("---- lane / time - Plot ----")
        #plt.plot(COM_x_new_time[0,:], COM_x_new_time[1,:])
        #plt.show()

        '''
        frames = pd.read_csv('../data/frames.csv', sep = ';')        
        lane_frames = []

        for lane in range(len(lanes)):
            if int(subject) not in list(frames['subject']):
                print('Subject ist not in frames.csv! Please add frames to the file.')
                break
            lane_start_end = []
            lane_start_end.append(frames['start'][(frames['test'] == test_type) & (frames['lane'] == lane+1) & (frames['subject'] == int(subject))].item())
            lane_start_end.append(frames['end'][(frames['test'] == test_type) & (frames['lane'] == lane+1) & (frames['subject'] == int(subject))].item())
            lane_frames.append(lane_start_end)
        '''
        _, lane_frames = get_frames(COM_x_new_time, sub_name, test_type)
        
        #print("lane_frames:", lane_frames)
        ### Define x,y,z for COM, LF and RF Data per Foot

        # exemplary dictionary:
        # com_data = {'lane_1': [[x_data][y_data][z_data]],
        #             'lane_2': [[x_data][y_data][z_data]]}

         
        com_data = {}
        lf_data = {}
        rf_data = {} 

        ## Build COM data dict per lane
        for lane in range(len(lanes)):
            lane_name = "lane_" + str(lane+1)
            x_data = COM_x_new_time[:, lane_frames[lane][0]:lane_frames[lane][1]]
            y_data = COM_y_time[:, lane_frames[lane][0]:lane_frames[lane][1]]
            z_data = COM_z_time[:, lane_frames[lane][0]:lane_frames[lane][1]]
            com_data[lane_name] = [x_data, y_data, z_data]


        ## Build Left/Right Foot data dict per lane
        for lane in range(len(lanes)):
            lane_name = "lane_" + str(lane+1)
            x_data = LeftFoot_x[lane_frames[lane][0]:lane_frames[lane][1]]
            y_data = LeftFoot_y[lane_frames[lane][0]:lane_frames[lane][1]]
            z_data = LeftFoot_z[lane_frames[lane][0]:lane_frames[lane][1]]
            lf_data[lane_name] = [x_data, y_data, z_data]

            x_data = RightFoot_x[lane_frames[lane][0]:lane_frames[lane][1]]
            y_data = RightFoot_y[lane_frames[lane][0]:lane_frames[lane][1]]
            z_data = RightFoot_z[lane_frames[lane][0]:lane_frames[lane][1]]
            rf_data[lane_name] = [x_data, y_data, z_data]

        
        var_all = []
        v_all = []

        Stride_length_all = [] #actually stride lengths
        Stride_length_all_L = [] 
        Stride_length_all_R = [] 

        step_width_all = []
        LF_height_sum = []
        RF_height_sum = []
        step_lengths = [] # step_lengths

        #swing & stances
        all_stances_L = []
        all_stances_R = []
        all_swings_L = []
        all_swings_R = []

        #percent swing and stances, currently calculated with all
        all_pc_st = []
        all_pc_sw = []
        
        #double support times for each lane in file
        dst_list = []

        for lane in lanes:
            #print(f'-------------------------{lane}-------------------------------')
            COM_xy_Lane_tp_new,  LF_xy_Lanetp_new, RF_xy_Lanetp_new = get_gait(lane, com_data, lf_data, rf_data, plot='show')

            var_lane = statistics.variance(COM_xy_Lane_tp_new[0,:])
            var_all.append(var_lane)
            
            v_lane = abs((COM_xy_Lane_tp_new[1,-1]-COM_xy_Lane_tp_new[1,0])/(com_data[lane][0][0,-1]-com_data[lane][0][0,0])) #(pos_end - pos_start) / (time_end - time_start)
            
            print("1, -1", COM_xy_Lane_tp_new[1,-1], len(COM_xy_Lane_tp_new[1, ]))
            print("0, 0 -1", com_data[lane][0][0,-1], len(com_data[lane][0][0,]))
            
            v_all.append(v_lane) #average of this is the speed
            
            peaks_COM, peaks_left, peaks_right, peaks_left_corrected, peaks_right_corrected, peaks_timeonly, peaks_heightonly, minima_RF, minima_LF = get_peaks(lane, com_data, lf_data, rf_data, plot = 'off') ##
            
            #STRIDE LENGTH
            stride_length, stride_length_L, stride_length_R = calculate_step_stride_length(lane, com_data, lf_data, rf_data)
            Stride_length_all += list(stride_length) #check this metric --> ACTUALLY STRIDE
            Stride_length_all_L += list(stride_length_L)
            Stride_length_all_R += list(stride_length_R)
            
            #STEP LENGTH
            step_lengths += list(get_step_lengths(minima_LF, minima_RF, LF_xy_Lanetp_new, RF_xy_Lanetp_new, lane)) #--> STEPS
            
            #STEP WIDTH
            step_width_all.append(calc_step_width(lane, com_data, lf_data, rf_data))

            LF_height_sum += list(peaks_left_corrected)
            RF_height_sum += list(peaks_right_corrected)
            
            #SWING AND STANCE TIME
            R, L, _, _ = intersect(lane, rf_data, lf_data, minima_RF, minima_LF, plot="off") ##
            swing_R, stance_R, swing_L, stance_L, dst_base = calculate_stance_swing(lane, L, R, lf_data, rf_data, com_data)
            dst = double_support_time(dst_base)
            pc_sw, pc_st = calculate_percent_cycle(swing_R, stance_R, swing_L, stance_L)
            
            dst_list.append(dst)
            
            all_stances_L += list(stance_L)
            all_stances_R += list(stance_R)
            all_swings_L += list(swing_L)
            all_swings_R += list(swing_R)
            
            all_pc_st.append(pc_st)
            all_pc_sw.append(pc_sw)
            
            plt.close()
            
        ## STANCE/SWING ##

        all_stances_L = np.array(all_stances_L).flatten() #flatten to 1d
        all_stances_R = np.array(all_stances_R).flatten()
        all_swings_L = np.array(all_swings_L).flatten()
        all_swings_R = np.array(all_swings_R).flatten()

        # average swing & stance, each leg (4)
        stance_L_avg = np.mean(all_stances_L)
        stance_R_avg = np.mean(all_stances_R)
        swing_L_avg = np.mean(all_swings_L)
        swing_R_avg = np.mean(all_swings_R)

        # average % swing and % stance 
        pc_st_avg = np.mean(all_pc_st)
        pc_sw_avg = np.mean(all_pc_sw)

        # TOTAL stance and swing percentages --> effectively the same
        total = np.sum(all_stances_L) + np.sum(all_stances_R) + np.sum(all_swings_L) + np.sum(all_swings_R)
        total_pc_sw = (np.sum(all_swings_L) + np.sum(all_swings_R)) / total
        total_pc_st = (np.sum(all_stances_L) + np.sum(all_stances_R)) / total

        # variability, both legs (1)
        stances_L_variability = variability(all_stances_L)
        stances_R_variability = variability(all_stances_R)
        swings_L_variability = variability(all_swings_L)
        swings_R_variability = variability(all_swings_R)

        # asymmetry, between legs (1)
        stances_asymmetry = asymmetry(all_stances_L, all_stances_R)
        swings_asymmetry = asymmetry(all_swings_L, all_swings_R)
        
        #double support time
        dst_avg = np.mean(dst_list)

        #interstep & frequency
        InterStepTime_avg, InterStepTime_arr = calc_interstep_all(lanes, com_data, lf_data, rf_data)
        Variability_InterStepTime = variability(InterStepTime_arr)
        Frequency = 1/InterStepTime_avg

        #### Calculate Variance of COMy and Velocity/Speed

        ## Durchschnittsgeschwindigkeit:
        variance_COMy = np.mean(var_all)
        speed = np.mean(v_all)

        #STRIDE
        stride_length_avg = np.mean(Stride_length_all) # Average STRIDE Length, all step lengths from right and left foot...
        stride_length_var = statistics.variance(Stride_length_all) # Variance
        stride_length_variability = variability(Stride_length_all)
        stride_length_asymmetry = asymmetry(Stride_length_all_L, Stride_length_all_R)

        #STEP -- NEW
        ac_step_length_avg = np.mean(step_lengths)
        ac_step_length_variance = statistics.variance(step_lengths)
        ac_step_length_variability = variability(step_lengths)

        step_width_all = [item for sublist in step_width_all for item in sublist] # macht aus einem nested array ein 1D-array
        #print(step_width_all)
        ##plt.plot(step_width_all, 'x')
        step_width_avg = np.mean(step_width_all)
        var_step_width = statistics.variance(step_width_all)
        step_width_variability = variability(step_width_all)

        #### Calculate Foot Height at Maximum Elevation
        LF_height_avg = np.mean(LF_height_sum)
        RF_height_avg = np.mean(RF_height_sum)

        foot_height_asymmetry = asymmetry(LF_height_sum, RF_height_sum)
        LF_height_variability = variability(LF_height_sum)
        RF_height_variability = variability(RF_height_sum)
        
        df = pd.DataFrame([trial_num,
                    test_type, 
                   stance_L_avg,
                   swing_L_avg,
                   stance_R_avg,
                   swing_R_avg,
                   stances_L_variability,
                   swings_L_variability,
                   stances_R_variability,
                   swings_R_variability,
                   stances_asymmetry,
                   swings_asymmetry,
                   total_pc_sw,
                   total_pc_st,
                   dst_avg,
                   InterStepTime_avg,
                   Variability_InterStepTime,
                   Frequency,
                   variance_COMy,
                   speed,
                   stride_length_avg,
                   stride_length_var,
                   stride_length_variability,
                   stride_length_asymmetry,
                   ac_step_length_avg,
                   ac_step_length_variance,
                   ac_step_length_variability,
                   step_width_avg,
                   var_step_width,
                   step_width_variability,
                   LF_height_avg,
                   RF_height_avg,
                   LF_height_variability,
                   RF_height_variability,
                   foot_height_asymmetry], index= INDICES, columns = [sub_name])

        all_results = pd.concat([all_results, df.T], axis = 0)
        all_results.to_csv('text_healthy_051926.csv')

        #df.to_csv(f'../export/{sub_name}.csv')
        #df.to_csv(f'out/{sub_name}.csv') 
        #print(f'Analysis done for subject: {sub_name}')
        #plt.close()
        counter += 1
        
    except Exception as e:
        print("file_strip", file_strip)
        print("sub_name", sub_name)
        print("trial_num", trial_num)
        
        print(f'Starting analysis for subject: {subject}')
        print(f"---------index {counter}---------")
        print(f'Exception occured: {e}')
        print(traceback.format_exc())
        error_subj.append(str(sub_name))
        error_trace.append(traceback.format_exc())
        error_idx.append(counter)
        counter += 1
        
        #printing plots only for debugging
        print("trying debugging plots")
        try:
            for lane in lanes:
                _ = get_peaks(lane, com_data, lf_data, rf_data, plot = 'off') ##
                _ = intersect(lane, rf_data, lf_data, minima_RF, minima_LF, plot = 'off') ##
        except Exception as e2:
            print(f'Exception occured: {e}')
            print(traceback.format_exc())

       
       
        

#print(results)
#results.to_csv(f'../export/results_walks.csv')
#results.to_excel(f'../export/results_walks.xlsx')


1, -1 3.9256558307546032 199
0, 0 -1 9.516666666666667 199
1, -1 0.8773195630798759 184
0, 0 -1 18.15 184
1, -1 3.922684670903915 169
0, 0 -1 24.5 169
1, -1 0.8811332406577757 166
0, 0 -1 31.283333333333335 166
1, -1 3.9259495261552546 163
0, 0 -1 37.233333333333334 163
1, -1 0.8891470616644797 163
0, 0 -1 43.63333333333333 163
1, -1 3.6040946700807237 225
0, 0 -1 9.716666666666667 225
1, -1 0.95526420830272 241
0, 0 -1 18.966666666666665 241
1, -1 3.6085244836495494 248
0, 0 -1 28.333333333333332 248
1, -1 0.9615976582990882 257
0, 0 -1 37.75 257
1, -1 3.9070950764380235 492
0, 0 -1 14.5 492
1, -1 1.0493495633493404 442
0, 0 -1 33.1 442
1, -1 3.867739722389381 474
0, 0 -1 50.05 474
1, -1 1.0353595223426186 498
0, 0 -1 66.76666666666667 498
1, -1 3.6766704719184014 218
0, 0 -1 10.7 218
1, -1 0.9341136043217991 229
0, 0 -1 18.466666666666665 229
1, -1 3.6843035355091858 221
0, 0 -1 25.966666666666665 221
1, -1 0.9377643833301768 185
0, 0 -1 32.43333333333333 185
1, -1 3.314703985540641 

/opt/miniconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/miniconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


1, -1 0.9964357882803306 208
0, 0 -1 23.5 208
1, -1 3.8781150263042257 208
0, 0 -1 31.083333333333332 208
1, -1 0.9949736068988695 199
0, 0 -1 38.05 199
1, -1 3.8719908974133577 214
0, 0 -1 45.53333333333333 214
1, -1 0.9961101045713254 198
0, 0 -1 52.36666666666667 198
1, -1 3.7667808446691105 128
0, 0 -1 6.216666666666667 128
1, -1 1.0559503914233137 124
0, 0 -1 10.816666666666666 124
1, -1 3.77291845752869 124
0, 0 -1 15.633333333333333 124
1, -1 1.0592462082965817 127
0, 0 -1 20.333333333333332 127
1, -1 3.7647139778494743 126
0, 0 -1 24.866666666666667 126
1, -1 1.0722920672610496 128
0, 0 -1 29.516666666666666 128
1, -1 3.76395126358902 178
0, 0 -1 13.35 178
1, -1 1.0046445773602026 161
0, 0 -1 18.866666666666667 161
1, -1 3.7534779634350777 167
0, 0 -1 24.683333333333334 167
1, -1 1.0001112110439103 167
0, 0 -1 30.5 167
1, -1 3.7566200769159255 170
0, 0 -1 36.21666666666667 170
1, -1 1.0096101252367111 172
0, 0 -1 42.016666666666666 172
1, -1 3.8861678543137703 271
0, 0 -1 12.6 

In [9]:
for i in all_results["double_support_time"]:
    print(i)

0.0404021644566128


In [5]:
all_results.to_csv('out/all_results_dst.csv')


In [8]:
len(error_subj)

28

In [6]:
demos = pd.read_csv("Demographics_PD.csv")

In [8]:
#subset to get rid of
demos_sub = demos.iloc[0:28, ]

demos_sub["ID"] = demos_sub["ID"].astype(int)
demos_sub["ID"] = demos_sub["ID"].astype(str)

#replace NV values with as control
demos_sub["Control"][14:16] = pd.Series(["1","1"])

#create mapping from control to subject ID
map = demos_sub[["ID", "Control"]]
map.set_index('ID', inplace=True)

mapping = map["Control"].to_dict()


demos_sub["Control"] = demos_sub["ID"].map(mapping)
all_results["ID"] = all_results.index
all_results["Control"] = all_results["ID"].map(mapping) #create Control column

/var/folders/sp/gln8rmf11713l0sb7h79sbf00000gn/T/ipykernel_58496/1655282577.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  demos_sub["ID"] = demos_sub["ID"].astype(int)
/var/folders/sp/gln8rmf11713l0sb7h79sbf00000gn/T/ipykernel_58496/1655282577.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  demos_sub["ID"] = demos_sub["ID"].astype(str)
/var/folders/sp/gln8rmf11713l0sb7h79sbf00000gn/T/ipykernel_58496/1655282577.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a D

In [9]:
all_results.to_csv("out/all_results_dst_ctrl.csv")